# Geração de amostra LHE: pp → γγ sem Higgs

Este notebook gera uma amostra MadGraph5_aMC@NLO com 10.000 eventos. O arquivo final será salvo como `data/fundo.lhe.gz` para ser usado pelos notebooks de análise Colab e SWAN.

O processo é gerado em nível partônico, sem detector, trigger ou pileup. A seção de choque e o processo registrado no LHE serão preservados para a análise posterior.

## 0. Localiza o projeto e prepara o diretório de dados

A célula procura a raiz do repositório mesmo quando o notebook está dentro de `notebooks/` ou quando o projeto foi descompactado no Colab. O resultado define a pasta `data/`, onde o arquivo LHE compactado será gravado.

In [ ]:
from pathlib import Path

def localizar_raiz():
    candidatos = [Path.cwd(), *Path.cwd().parents]
    for candidato in candidatos:
        if (candidato / 'data').is_dir():
            return candidato.resolve()
        for pasta in candidato.glob('*-main'):
            if (pasta / 'data').is_dir():
                return pasta.resolve()
    raise FileNotFoundError('Não encontrei a pasta data/. Abra o notebook a partir do projeto analise.')

RAIZ = localizar_raiz()
PASTA_DADOS = RAIZ / 'data'
PASTA_DADOS.mkdir(parents=True, exist_ok=True)
MG5_DIR = RAIZ / 'MG5_aMC_v3_8_1'
MG5_URL = 'https://launchpad.net/mg5amcnlo/3.0/3.8.x/+download/MG5_aMC_v3.8.1.tar.gz'
print(f'Raiz do projeto: {RAIZ}')
print('Saída:', PASTA_DADOS / 'fundo.lhe.gz')


## 1. Baixa e prepara o MadGraph5_aMC@NLO

O código usa apenas a biblioteca padrão do Python para baixar e descompactar a versão 3.8.1 do MadGraph, caso ela ainda não esteja presente. Em uma execução posterior, o download é ignorado e a mesma instalação é reutilizada.

In [ ]:
import tarfile
import urllib.request

def preparar_mg5():
    executavel = MG5_DIR / 'bin' / 'mg5_aMC'
    if executavel.exists():
        print(f'MadGraph já disponível: {executavel}')
        return executavel
    arquivo = RAIZ / 'MG5_aMC_v3.8.1.tar.gz'
    print(f'Baixando {MG5_URL}')
    urllib.request.urlretrieve(MG5_URL, arquivo)
    with tarfile.open(arquivo, 'r:gz') as tar:
        tar.extractall(RAIZ)
    if not executavel.exists():
        raise FileNotFoundError(f'MadGraph não foi encontrado em {executavel}')
    return executavel

MG5 = preparar_mg5()


## 2. Escreve o cartão e gera os 10.000 eventos

O cartão usa o modelo `sm` e o processo `generate p p > a a / h`. A saída é criada em uma pasta de trabalho do MG5; a célula seguinte localizará o arquivo `unweighted_events.lhe.gz` produzido pelo lançamento.

In [ ]:
import subprocess

PROCESSO_ID = 'pp_gamma_gamma'
PROCESSO_MG5 = 'generate p p > a a / h'
MODELO_MG5 = 'sm'
PASTA_MG5 = RAIZ / 'mg5_runs' / PROCESSO_ID
CARD = RAIZ / f'geracao_{PROCESSO_ID}.mg5'
CARD.write_text(
    '\n'.join([
        f'import model {MODELO_MG5}',
        PROCESSO_MG5,
        f'output {PASTA_MG5}',
        'y',
        'launch',
        'set nevents 10000',
        'set small_width_treatment 1e-12',
        'set use_syst false',
        'done',
    ]) + '\n',
    encoding='utf-8',
)
print(CARD.read_text())
subprocess.run([str(MG5), str(CARD)], cwd=RAIZ, check=True)


## 3. Copia e valida o LHE na pasta `data/`

A amostra gerada é compactada sem alterar o conteúdo. A validação conta os blocos `<event>` e verifica se o arquivo contém exatamente 10.000 eventos antes de concluir.

In [ ]:
import gzip
import shutil

candidatos = list(PASTA_MG5.glob('Events/**/unweighted_events.lhe.gz'))
if not candidatos:
    raise FileNotFoundError(f'Nenhum LHE encontrado em {PASTA_MG5}')
origem = max(candidatos, key=lambda caminho: caminho.stat().st_mtime)
destino = PASTA_DADOS / 'fundo.lhe.gz'
shutil.copy2(origem, destino)
with gzip.open(destino, 'rt', encoding='utf-8') as arquivo:
    eventos = sum(1 for linha in arquivo if linha.strip() == '<event>')
print(f'LHE salvo em: {destino}')
print(f'Eventos encontrados: {eventos}')
if eventos != 10000:
    raise ValueError(f'Esperava 10000 eventos, mas encontrei {eventos}')


## Resultado

O arquivo `data/fundo.lhe.gz` está pronto para ser usado nos notebooks de análise. Depois da geração, confira também a seção `<init>` para registrar a seção de choque e o processo no relatório da análise.